# 04 – Kenntage und Extreme aus Tageswerten

Aus den Tageswerten der DWD-Stationen berechnen wir je Station und Jahr **Klimakenntage** (Sommertage, heiße Tage, Tropennächte, Frost- und Eistage, Hitzewellen) und **Niederschlagsindizes** (Starkregentage, größte Tages- und 5-Tages-Summen, Trockenperioden). Für Deutschland vergleichen wir mit den offiziellen DWD-Gebietsmitteln; einzelne Stationen weltweit kommen aus **GHCN-Daily**.

Module: `src/klima/kenntage.py`, `src/klima/niederschlag.py`; Schwellen in `konfiguration/analyse.toml`.

In [ ]:
%matplotlib inline
import pandas as pd

from klima import auswertungen, einlesen
from klima.grafik import statisch
from klima.kenntage import temperaturindizes
from klima.niederschlag import niederschlagsindizes

pd.set_option("display.float_format", "{:.2f}".format)

## Je Station und Jahr – Beispiel Potsdam

In [ ]:
potsdam = einlesen.dwd_tageswerte(stationen=["03987"])
kenntage = temperaturindizes(potsdam[["stations_id", "datum", "jahr", "tmax", "tmin"]])
kenntage.set_index("jahr").loc[
    2015:2025,
    [
        "sommertage",
        "heisse_tage",
        "tropennaechte",
        "hitzewellen",
        "laengste_hitzewelle",
        "frosttage",
    ],
]

In [ ]:
regen = niederschlagsindizes(potsdam[["stations_id", "datum", "jahr", "niederschlag"]])
regen.set_index("jahr").loc[
    2015:2025, ["jahressumme", "starkniederschlag_10mm", "rx1day", "rx5day", "cdd", "r95p_anteil"]
]

## Deutschland: eigene Berechnung vs. DWD

Gebietsmittel je Index: Abweichung jeder Station von ihrem Mittel 1951–1980 → 1°-Gitter → Flächenmittel + gegittertes Referenzmittel. Dauert rund eine Minute (alle ~19 Mio. Tageswerte).

In [ ]:
ergebnis = auswertungen.kenntage_deutschland()
pd.DataFrame(
    {
        name: {"Trend pro Dekade": t.steigung_pro_dekade, "± 95 %": t.unsicherheit_95}
        for name, t in ergebnis.trends.items()
    }
).T

In [ ]:
offiziell = einlesen.dwd_gebietsmittel_kenntage().pivot(
    index="jahr", columns="kenngroesse", values="wert"
)
eigen = ergebnis.tabellen["deutschland_temperatur"].join(
    ergebnis.tabellen["deutschland_niederschlag"]
)
vergleich = {}
for spalte in [
    "sommertage",
    "heisse_tage",
    "tropennaechte",
    "frosttage",
    "eistage",
    "starkniederschlag_10mm",
    "starkniederschlag_20mm",
]:
    differenz = (eigen[spalte] - offiziell[spalte].astype(float)).dropna()
    vergleich[spalte] = {
        "Korrelation": eigen[spalte].corr(offiziell[spalte].astype(float)),
        "mittlere Differenz": differenz.mean(),
    }
pd.DataFrame(vergleich).T

Die eigene Berechnung trifft die offiziellen Werte sehr gut. Systematisch zählen wir etwas **weniger Sommertage** (fehlende Tage werden nicht hochgerechnet) und etwas **mehr Starkregentage** – das 1-km-Raster des DWD glättet einzelne Extremwerte von Stationen.

In [ ]:
statisch.zeitreihen(
    ergebnis.tabellen["tag_nacht"],
    "Tage oder Nächte – was erwärmt sich schneller?",
    referenz=(1951, 1980),
);

## Stationen weltweit (GHCN-Daily)

Stationen suchen: `uv run klima ghcnd-suchen --land US --name "NY CITY"`, laden: `uv run klima laden ghcnd_tageswerte --station USW00094728`.

In [ ]:
einlesen.ghcnd_tageswerte(spalten=["stations_id", "datum"]).groupby(
    "stations_id", observed=True
).datum.agg(["min", "max", "size"])

In [ ]:
new_york = auswertungen.kenntage_station("USW00094728")
{name: t.text(new_york.einheiten[name]) for name, t in new_york.trends.items()}

Die Tagesspanne in New York springt um 1910–1920 um rund 1,5 °C – typisch für eine **Inhomogenität** (Verlegung oder neues Messgerät), nicht für eine Klimaänderung. GHCN-Daily ist nicht homogenisiert; Trends einzelner Stationen sind daher mit Vorsicht zu lesen.